
# CardioVision 3D
## Cardiovascular Risk Visualization & Prediction

### 1. Project Overview
CardioVision 3D is a data analytics and machine learning
project focused on analyzing clinical patient data to
predict Coronary Artery Disease (CAD) and vessel-specific
stenosis probabilities.

### 2. Project Objectives
- Understand the cardiovascular dataset.
- Perform data cleaning and preprocessing.
- Conduct Exploratory Data Analysis (EDA).
- Perform statistical analysis.
- Build machine learning classification models.
- Explain model predictions using SHAP.
- Develop an interactive 3D heart visualization.
- Build a full-stack web application.

### 3. Technology Stack
- Python
- Pandas and NumPy
- Matplotlib and Seaborn
- Scikit-learn and XGBoost
- PostgreSQL
- FastAPI
- React and TypeScript
- Three.js

### 4. Current Notebook
Notebook: 01_dataset_understanding
Purpose:
To understand the dataset structure, features,
target variables, and data quality before analysis.

### 5. Important Note
This project is an educational and research prototype.
Its predictions are not medical diagnoses.


In [ ]:

import pandas as pd

# Path to original Excel dataset
file_path = "../dataset/raw/extention of Z-Alizadeh sani dataset.xlsx"

# Load dataset
df = pd.read_excel(file_path)

print("Dataset loaded successfully!")


In [ ]:

# Check dataset dimensions

rows, columns = df.shape

print("Total Patient Records:", rows)
print("Total Features:", columns)


In [ ]:

# Display all column names

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")


In [ ]:

# Display columns 26 to 55

for index, column in enumerate(df.columns[25:55], start=26):
    print(f"{index}. {column}")


In [ ]:

# Identify the missing column

print("Column 51:", df.columns[50])


In [ ]:
# Verify all 59 columns and their data types

column_info = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

display(column_info)


In [ ]:

# Check missing values in every column

missing_values = df.isnull().sum()

print("Missing values per column:")
display(missing_values[missing_values > 0])

print("Total missing values:", df.isnull().sum().sum())


In [ ]:

# Check duplicate rows

duplicate_count = df.duplicated().sum()

print("Total duplicate rows:", duplicate_count)


In [ ]:

# Inspect categorical columns and their unique values

categorical_columns = df.select_dtypes(
    include=["object", "str"]
).columns

for column in categorical_columns:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))


In [ ]:

# Investigate Sex column values

print("Unique values:", df["Sex"].unique())

print("\nValue counts:")
print(df["Sex"].value_counts(dropna=False))

print("\nData type:", df["Sex"].dtype)


In [ ]:

# Inspect remaining categorical columns
# Excluding Sex, which we already investigated

for column in categorical_columns:
    if column != "Sex":
        print(f"\n{'='*40}")
        print(f"Column: {column}")
        print(df[column].value_counts(dropna=False))


In [ ]:
df.describe().T

In [ ]:
# Identify potential outliers using IQR

numeric_df = df.select_dtypes(include="number")

Q1 = numeric_df.quantile(0.25)
Q3 = numeric_df.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_count = (
    (numeric_df < lower_bound) |
    (numeric_df > upper_bound)
).sum()

outlier_count[outlier_count > 0].sort_values(ascending=False)

In [ ]:
# Identify binary numerical columns

binary_columns = [
    col for col in numeric_df.columns
    if numeric_df[col].nunique() <= 2
]

continuous_columns = [
    col for col in numeric_df.columns
    if numeric_df[col].nunique() > 2
]

print("Binary columns:", binary_columns)
print("\nContinuous columns:", continuous_columns)

In [ ]:
# IQR analysis only on continuous/multi-valued columns

analysis_df = numeric_df[continuous_columns]

Q1 = analysis_df.quantile(0.25)
Q3 = analysis_df.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_count = (
    (analysis_df < lower_bound) |
    (analysis_df > upper_bound)
).sum()

outlier_count[outlier_count > 0].sort_values(ascending=False)

In [ ]:
# Inspect IQR boundaries for selected columns

selected_columns = ["FBS", "TG", "BUN", "BMI", "BP"]

outlier_summary = pd.DataFrame({
    "Q1": Q1[selected_columns],
    "Q3": Q3[selected_columns],
    "IQR": IQR[selected_columns],
    "Lower Bound": lower_bound[selected_columns],
    "Upper Bound": upper_bound[selected_columns],
    "Minimum": analysis_df[selected_columns].min(),
    "Maximum": analysis_df[selected_columns].max()
})

outlier_summary

In [ ]:
# Inspect actual outlier records in selected columns

selected_columns = ["FBS", "TG", "BUN", "BMI", "BP"]

for col in selected_columns:
    mask = (
        (df[col] < lower_bound[col]) |
        (df[col] > upper_bound[col])
    )

    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(df.loc[mask, [col]].to_string())

In [ ]:
# Display all FBS outlier records clearly

col = "FBS"

mask = (
    (df[col] < lower_bound[col]) |
    (df[col] > upper_bound[col])
)

fbs_outliers = df.loc[mask, [col]]

print("Total FBS outliers:", len(fbs_outliers))
display(fbs_outliers)

In [ ]:
col = "TG"

mask = (
    (df[col] < lower_bound[col]) |
    (df[col] > upper_bound[col])
)

tg_outliers = df.loc[mask, [col]]

print("Total TG outliers:", len(tg_outliers))
display(tg_outliers)

In [ ]:
col = "BUN"

mask = (
    (df[col] < lower_bound[col]) |
    (df[col] > upper_bound[col])
)

bun_outliers = df.loc[mask, [col]]

print("Total BUN outliers:", len(bun_outliers))
display(bun_outliers)

In [ ]:
col = "BMI"

mask = (
    (df[col] < lower_bound[col]) |
    (df[col] > upper_bound[col])
)

bmi_outliers = df.loc[mask, [col]]

print("Total BMI outliers:", len(bmi_outliers))
display(bmi_outliers)

In [ ]:

col = "BP"

mask = (
    (df[col] < lower_bound[col]) |
    (df[col] > upper_bound[col])
)

bp_outliers = df.loc[mask, [col]]

print("Total BP outliers:", len(bp_outliers))
display(bp_outliers)

In [ ]:
# Create a summary of potential outliers

outlier_summary = pd.DataFrame({
    "Outlier Count": outlier_count,
    "Percentage": (
        outlier_count / len(df) * 100
    ).round(2)
})

outlier_summary = outlier_summary[
    outlier_summary["Outlier Count"] > 0
].sort_values("Outlier Count", ascending=False)

display(outlier_summary)

In [ ]:
# Inspect ordinal / coded numerical variables

for col in ["Function Class", "Region RWMA"]:
    print(f"\nColumn: {col}")
    print("Unique values:", sorted(df[col].unique()))
    print(df[col].value_counts().sort_index())

In [ ]:
# Create a basic dataset profile

data_profile = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Unique Values": df.nunique(),
    "Missing Values": df.isnull().sum(),
    "Unique Samples": [
        df[col].dropna().unique()[:5].tolist()
        for col in df.columns
    ]
})

display(data_profile)

In [ ]:
# Identify columns having only one unique value

constant_columns = [
    col for col in df.columns
    if df[col].nunique() == 1
]

print("Constant columns:", constant_columns)

for col in constant_columns:
    print(f"{col}: {df[col].unique()}")

In [ ]:
# Investigate Sex column values

print("Unique values:", df["Sex"].unique())

print("\nValue counts:")
print(df["Sex"].value_counts())

print("\nRaw values with whitespace check:")
print([repr(value) for value in df["Sex"].unique()])

In [ ]:
# Document data quality observations

quality_notes = [
    {
        "Column": "Sex",
        "Issue": "Possible spelling inconsistency",
        "Observation": "Male = 176, Fmale = 127",
        "Decision": "Verify before correction"
    },
    {
        "Column": "Exertional CP",
        "Issue": "Constant column",
        "Observation": "All 303 records contain N",
        "Decision": "Consider excluding from model features"
    },
    {
        "Column": "Function Class",
        "Issue": "Coded ordinal variable",
        "Observation": "Values: 0, 1, 2, 3",
        "Decision": "Investigate category meaning"
    },
    {
        "Column": "Region RWMA",
        "Issue": "Coded variable",
        "Observation": "Values: 0, 1, 2, 3, 4",
        "Decision": "Investigate category meaning"
    },
    {
        "Column": "Numerical Features",
        "Issue": "Potential outliers",
        "Observation": "IQR method identified unusual observations",
        "Decision": "Investigate before cleaning"
    }
]

quality_report = pd.DataFrame(quality_notes)

display(quality_report)

In [ ]:
import pandas as pd

print("Pandas version:", pd.__version__)

In [ ]:
quality_report = pd.DataFrame(quality_notes)

display(quality_report)

In [ ]:
quality_report.to_csv(
    "../reports/data_quality_report.csv",
    index=False
)

print("Data Quality Report saved successfully!")

In [ ]:
# Create a basic data dictionary

data_dictionary = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Unique Values": df.nunique().values,
    "Missing Values": df.isnull().sum().values,
    "Sample Values": [
        ", ".join(map(str, df[col].dropna().unique()[:5]))
        for col in df.columns
    ]
})

display(data_dictionary.head(10))

In [ ]:
import pandas as pd

file_path = "../dataset/raw/extention of Z-Alizadeh sani dataset.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded:", df.shape)

In [ ]:
# Create a basic data dictionary

data_dictionary = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Unique Values": df.nunique().values,
    "Missing Values": df.isnull().sum().values,
    "Sample Values": [
        ", ".join(map(str, df[col].dropna().unique()[:5]))
        for col in df.columns
    ]
})

display(data_dictionary.head(10))

In [ ]:
data_dictionary.to_csv(
    "../data_dictionary/data_dictionary.csv",
    index=False
)

print("Data Dictionary saved successfully!")
print("Total columns documented:", len(data_dictionary))

In [ ]:
# Inspect all four prediction targets

target_columns = ["Cath", "LAD", "LCX", "RCA"]

for col in target_columns:
    print(f"\n{'='*35}")
    print(f"Target: {col}")
    print(df[col].value_counts())
    print("\nPercentage:")
    print(
        (df[col].value_counts(normalize=True) * 100).round(2)
    )

In [ ]:
# Create a compact target distribution summary

target_columns = ["Cath", "LAD", "LCX", "RCA"]

target_summary = pd.concat(
    [
        df[col].value_counts().rename(f"{col}_Count")
        for col in target_columns
    ],
    axis=1
).fillna(0).astype(int)

print("Target Distribution Summary:")
display(target_summary)

In [ ]:
import matplotlib.pyplot as plt

target_columns = ["Cath", "LAD", "LCX", "RCA"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flatten(), target_columns):
    percentages = df[col].value_counts(normalize=True) * 100

    percentages.plot(
        kind="bar",
        ax=ax,
        title=f"{col} Distribution",
        rot=0
    )

    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel("Class")

    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

target_columns = ["Cath", "LAD", "LCX", "RCA"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flatten(), target_columns):
    percentages = df[col].value_counts(normalize=True) * 100

    percentages.plot(
        kind="bar",
        ax=ax,
        title=f"{col} Distribution",
        rot=0
    )

    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel("Class")

    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%")

plt.tight_layout()
plt.show()

In [ ]:
plt.savefig(
    "../reports/target_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.tight_layout()

plt.savefig(
    "../reports/target_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

target_columns = ["Cath", "LAD", "LCX", "RCA"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flatten(), target_columns):
    percentages = df[col].value_counts(normalize=True) * 100

    percentages.plot(
        kind="bar",
        ax=ax,
        title=f"{col} Distribution",
        rot=0
    )

    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel("Class")

    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%")

plt.tight_layout()

plt.savefig(
    "../reports/target_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Define prediction target columns
target_columns = ["Cath", "LAD", "LCX", "RCA"]

# Select numerical input features only
numeric_features = df.select_dtypes(
    include=["int64", "float64"]
).drop(columns=target_columns, errors="ignore")

print("Total numerical input features:", len(numeric_features.columns))
print("\nSelected features:")
print(numeric_features.columns.tolist())

In [ ]:
# Calculate Pearson correlation

correlation_matrix = numeric_features.corr(
    method="pearson"
)

print("Correlation matrix shape:", correlation_matrix.shape)

display(correlation_matrix.round(2))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(18, 14))

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    linewidths=0.3
)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()

plt.savefig(
    "../reports/feature_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(18, 14))

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    linewidths=0.3
)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()

plt.savefig(
    "../reports/feature_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Identify highly correlated feature pairs

import numpy as np

upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

high_correlation = (
    upper_triangle
    .stack()
    .reset_index()
)

high_correlation.columns = [
    "Feature 1",
    "Feature 2",
    "Correlation"
]

high_correlation = high_correlation[
    high_correlation["Correlation"].abs() >= 0.70
]

high_correlation = high_correlation.sort_values(
    by="Correlation",
    key=abs,
    ascending=False
)

display(high_correlation)

In [ ]:
high_correlation.to_csv(
    "../reports/high_correlation_pairs.csv",
    index=False
)

print("High correlation report saved!")

In [ ]:
# Select categorical input features

categorical_features = df.select_dtypes(
    include=["object", "str"]
).drop(
    columns=["LAD", "LCX", "RCA", "Cath"],
    errors="ignore"
)

print("Total categorical input features:",
      len(categorical_features.columns))

print("\nCategorical columns:")
print(categorical_features.columns.tolist())

In [ ]:
# Analyze category counts and percentages

for column in categorical_features.columns:

    print(f"\n{'='*45}")
    print(f"Column: {column}")

    counts = categorical_features[column].value_counts(
        dropna=False
    )

    percentages = (
        categorical_features[column]
        .value_counts(normalize=True, dropna=False) * 100
    )

    summary = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages.round(2)
    })

    display(summary)

categorical_quality = []

for column in categorical_features.columns:

    counts = categorical_features[column].value_counts()

    categorical_quality.append({
        "Column": column,
        "Unique Categories": categorical_features[column].nunique(),
        "Rare Categories (<5%)": [
            category
            for category, count in counts.items()
            if (count / len(df)) * 100 < 5
        ],
        "Constant Column": counts.size == 1
    })

categorical_quality_report = pd.DataFrame(categorical_quality)

display(categorical_quality_report)

In [ ]:
categorical_quality = []

for column in categorical_features.columns:

    counts = categorical_features[column].value_counts()

    categorical_quality.append({
        "Column": column,
        "Unique Categories": categorical_features[column].nunique(),
        "Rare Categories (<5%)": [
            category
            for category, count in counts.items()
            if (count / len(df)) * 100 < 5
        ],
        "Constant Column": counts.size == 1
    })

categorical_quality_report = pd.DataFrame(categorical_quality)

display(categorical_quality_report)

In [ ]:
categorical_quality_report.to_csv(
    "../reports/categorical_quality_report.csv",
    index=False
)

print("Categorical Quality Report saved successfully!")

In [ ]:
print("Categorical Features:", len(categorical_features.columns))

print(
    "Constant Columns:",
    categorical_quality_report[
        categorical_quality_report["Constant Column"]
    ]["Column"].tolist()
)

print(
    "Features with Rare Categories:",
    (categorical_quality_report["Rare Categories (<5%)"].apply(len) > 0).sum()

)

# CardioVision 3D — Exploratory Data Analysis (EDA)

## 1. Dataset Overview
- **Dataset Name:** Extension of Z-Alizadeh Sani Dataset
- **Source:** UCI Machine Learning Repository
- **Total Records:** 303
- **Total Features:** 59
- **Missing Values:** 0
- **Duplicate Records:** 0
- **Problem Type:** Classification
- **Target Variables:** Cath, LAD, LCX, RCA

## 2. Data Quality Summary

- **Missing Values:** No missing values found.
- **Duplicate Records:** No duplicate rows found.
- **Constant Feature:** `Exertional CP` contains only one unique value (`N`).
- **Rare Categories:** 12 categorical features contain categories with less than 5% frequency.
- **Potential Spelling Issue:** The `Sex` column contains `Fmale`. This requires verification before correction.
- **Outliers:** Several numerical features contain potential outliers based on the IQR method.

## 3. Categorical Feature Analysis

- **Total Categorical Features:** 21
- **Constant Feature:** `Exertional CP`
- **Rare Categories:** 12 features contain categories with less than 5% frequency.
- **Multi-category Features:** `BBB` and `VHD` contain multiple categories.
- **Class Imbalance:** Some categorical values occur very infrequently, such as `CHF = Y` and `LowTH Ang = Y`.

**Observation:** Rare categories and class distributions should be considered during preprocessing and model evaluation.

## 4. Numerical Feature Analysis

- **Outlier Detection Method:** Interquartile Range (IQR).
- **Potential Outliers:** Multiple numerical features contain values outside the IQR boundaries.
- **Features Requiring Further Investigation:** FBS, TG, BUN, BP, BMI, and other numerical variables.
- **Special Cases:** `Function Class` and `Region RWMA` are coded variables and should not be treated as ordinary continuous measurements.

**Observation:** Potential outliers require domain-aware investigation before deciding whether any preprocessing is necessary.

## 5. Feature Correlation Analysis

- **Correlation Method:** Pearson correlation.
- **Correlation Threshold:** Absolute correlation value >= 0.70.
- **Strong Negative Correlation:** `Lymph` and `Neut` (-0.923).
- **Strong Positive Correlation:** `Weight` and `BMI` (+0.725).
- **Feature Removal:** No features have been removed based on correlation yet.

**Observation:** Strongly correlated features may contain overlapping information. Their impact should be evaluated during model development.

## 6. Target Variable Analysis

The dataset contains four target variables:

- **Cath:** CAD (216 records), Normal (87 records).
- **LAD:** Stenotic (177 records), Normal (126 records).
- **LCX:** Stenotic (119 records), Normal (184 records).
- **RCA:** Stenotic (114 records), Normal (189 records).

**Observation:** The target variables have different class distributions. Class imbalance should be considered during model training and evaluation.

## 7. Initial EDA Conclusions

- The dataset contains 303 records and 59 features.
- No missing or duplicate records were found.
- One constant categorical feature was identified.
- Rare categories and potential numerical outliers require attention.
- Two strongly correlated feature pairs were identified.
- All four target variables have been examined.
- No original data has been modified or deleted.

**Next Phase:** Data preprocessing and feature preparation.




In [ ]:
df[["Weight", "Length", "BMI"]].head(10)

In [ ]:
import pandas as pd

file_path = "../dataset/raw/extention of Z-Alizadeh sani dataset.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df[["Weight", "Length", "BMI"]].head(10)